# 03 — Prétraitement des données

## Projet : Smart City Energy Forecasting — Tetouan

Ce notebook correspond à la phase de prétraitement du pipeline KDD.

Après le Data Audit et l’EDA, l’objectif est de transformer le fichier brut en un dataset propre, régulier et exploitable pour les prochaines étapes :

- feature engineering ;
- modélisation ;
- évaluation ;
- interprétation ;
- valorisation métier.

Les opérations principales sont :

1. chargement du fichier brut ;
2. nettoyage et renommage des colonnes ;
3. conversion de la colonne temporelle ;
4. contrôle de la régularité temporelle ;
5. création de la cible `target` ;
6. création de la variable `total_load` ;
7. rééchantillonnage horaire ;
8. gestion sécurisée des valeurs manquantes ;
9. contrôle des valeurs incohérentes ;
10. détection des anomalies ;
11. split chronologique train / validation / test ;
12. sauvegarde des datasets propres.

**Version corrigée :** cette version corrige les `NaN` créés par les Rolling Z-Scores et supprime le risque de data leakage dans les variables de scaling de base.

## 1. Importation des bibliothèques

Cette cellule importe les bibliothèques nécessaires au prétraitement.

`pandas` est utilisé pour manipuler les données temporelles.  
`numpy` est utilisé pour les calculs numériques.  
`Path` permet de gérer proprement les chemins du projet.  
`StandardScaler` est utilisé pour préparer une normalisation sans fuite de données.  
`joblib` permet de sauvegarder les scalers.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

from sklearn.preprocessing import StandardScaler
import joblib
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Définition des chemins du projet

Le fichier brut est conservé dans `data/raw`.

Les fichiers prétraités seront sauvegardés dans `data/processed`.

Les objets nécessaires aux modèles, comme les scalers, seront sauvegardés dans `models`.

Un rapport JSON de preprocessing sera sauvegardé dans `results/preprocessing`.

In [2]:
DATA_FILENAME = "Tetuan City power consumption.csv"

def find_project_root(data_filename: str) -> Path:
    """
    Recherche automatiquement la racine du projet.

    Le notebook peut être exécuté :
    - depuis le dossier notebooks/ ;
    - depuis la racine du projet ;
    - depuis un environnement temporaire où le CSV est placé directement dans le dossier courant.

    La fonction accepte deux organisations :
    1. PROJECT_ROOT/data/raw/Tetuan City power consumption.csv
    2. PROJECT_ROOT/Tetuan City power consumption.csv
    """
    cwd = Path.cwd().resolve()

    for base in [cwd, *cwd.parents]:
        raw_candidate = base / "data" / "raw" / data_filename
        direct_candidate = base / data_filename

        if raw_candidate.exists() or direct_candidate.exists():
            return base

    raise FileNotFoundError(
        "Impossible de localiser le fichier brut. "
        "Placez le CSV dans data/raw/ ou exécutez le notebook depuis la racine du projet."
    )

PROJECT_ROOT = find_project_root(DATA_FILENAME)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / DATA_FILENAME
if not RAW_PATH.exists():
    RAW_PATH = PROJECT_ROOT / DATA_FILENAME

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results" / "preprocessing"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Fichier introuvable : {RAW_PATH}")

print(f"Racine du projet détectée : {PROJECT_ROOT}")
print(f"Fichier brut trouvé       : {RAW_PATH}")
print(f"Dossier processed         : {PROCESSED_DIR}")
print(f"Dossier models            : {MODELS_DIR}")
print(f"Dossier results           : {RESULTS_DIR}")

Racine du projet détectée : D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan
Fichier brut trouvé       : D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\raw\Tetuan City power consumption.csv
Dossier processed         : D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed
Dossier models            : D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\models
Dossier results           : D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\results\preprocessing


## 3. Définition du mapping des colonnes

Les noms originaux du dataset contiennent des majuscules, des espaces et parfois des doubles espaces.

Pour éviter les erreurs dans le code, les colonnes sont renommées avec des noms techniques simples :

- `DateTime` devient `datetime` ;
- `Temperature` devient `temperature` ;
- `Wind Speed` devient `wind_speed` ;
- `Zone 1 Power Consumption` devient `zone1_power`.

Cette standardisation est essentielle pour construire un pipeline propre et réutilisable.

In [3]:
COLUMN_MAPPING = {
    "DateTime": "datetime",
    "Temperature": "temperature",
    "Humidity": "humidity",
    "Wind Speed": "wind_speed",
    "general diffuse flows": "general_diffuse_flows",
    "diffuse flows": "diffuse_flows",
    "Zone 1 Power Consumption": "zone1_power",
    "Zone 2 Power Consumption": "zone2_power",
    "Zone 3 Power Consumption": "zone3_power"
}

NUMERIC_COLS = [
    "temperature",
    "humidity",
    "wind_speed",
    "general_diffuse_flows",
    "diffuse_flows",
    "zone1_power",
    "zone2_power",
    "zone3_power"
]

EXPECTED_COLS = ["datetime"] + NUMERIC_COLS

## 4. Chargement du fichier brut

Cette cellule charge le fichier CSV original.

Le fichier brut ne doit pas être modifié directement. Toutes les transformations sont appliquées dans le notebook et les résultats sont sauvegardés dans `data/processed`.

In [4]:
df_raw = pd.read_csv(RAW_PATH)

print(f"Dimensions brutes : {df_raw.shape}")
print("Colonnes originales :")
print(df_raw.columns.tolist())

display(df_raw.head())

Dimensions brutes : (52416, 9)
Colonnes originales :
['DateTime', 'Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows', 'Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']


,DateTime,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
0,1/1/2017 0:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
1,1/1/2017 0:10,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2,1/1/2017 0:20,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373
3,1/1/2017 0:30,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711
4,1/1/2017 0:40,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964


## 5. Nettoyage des noms de colonnes

Avant le renommage, les noms de colonnes sont nettoyés :

- suppression des espaces au début et à la fin ;
- remplacement des espaces multiples par un seul espace.

Cette étape rend le pipeline plus robuste, notamment pour les colonnes `Zone 2` et `Zone 3`, qui peuvent contenir des doubles espaces.

In [5]:
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Colonnes après nettoyage des espaces :")
print(df.columns.tolist())

Colonnes après nettoyage des espaces :
['DateTime', 'Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows', 'Zone 1 Power Consumption', 'Zone 2 Power Consumption', 'Zone 3 Power Consumption']


## 6. Renommage des colonnes

Les colonnes sont renommées selon le mapping défini précédemment.

Une vérification est ensuite effectuée pour s’assurer que toutes les colonnes attendues sont présentes.

In [6]:
df = df.rename(columns=COLUMN_MAPPING)

missing_cols = [col for col in EXPECTED_COLS if col not in df.columns]

if missing_cols:
    raise ValueError(f"Colonnes manquantes après renommage : {missing_cols}")

print("Renommage terminé avec succès.")
print("Colonnes finales :")
print(df.columns.tolist())

Renommage terminé avec succès.
Colonnes finales :
['datetime', 'temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows', 'zone1_power', 'zone2_power', 'zone3_power']


## 7. Conversion de la colonne temporelle

La colonne `datetime` est convertie en véritable format temporel.

Le format du dataset est :

`mois/jour/année heure:minute`

Exemple :

`1/1/2017 0:00`

La conversion est contrôlée afin de détecter d’éventuelles dates invalides.

In [7]:
df["datetime"] = pd.to_datetime(
    df["datetime"],
    format="%m/%d/%Y %H:%M",
    errors="coerce"
)

invalid_dates = df["datetime"].isna().sum()

print(f"Dates non convertibles : {invalid_dates}")

if invalid_dates > 0:
    raise ValueError(f"{invalid_dates} dates n'ont pas pu être converties.")

Dates non convertibles : 0


## 8. Conversion des variables numériques

Les colonnes météorologiques et les colonnes de consommation doivent être numériques.

Cette cellule force la conversion numérique et compte les éventuelles valeurs non convertibles.

In [8]:
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_missing = df[NUMERIC_COLS].isna().sum()

print("Valeurs manquantes après conversion numérique :")
display(numeric_missing.to_frame(name="missing_count"))

total_numeric_missing = numeric_missing.sum()

if total_numeric_missing > 0:
    print(f"Attention : {total_numeric_missing} valeurs numériques manquantes détectées.")
else:
    print("Toutes les variables numériques sont correctement converties.")

Valeurs manquantes après conversion numérique :


,missing_count
temperature,0
humidity,0
wind_speed,0
general_diffuse_flows,0
diffuse_flows,0
zone1_power,0
zone2_power,0
zone3_power,0


Toutes les variables numériques sont correctement converties.


## 9. Audit temporel avant indexation

Avant de définir `datetime` comme index, on vérifie :

- la période couverte ;
- les doublons temporels ;
- l’ordre chronologique ;
- le nombre de valeurs manquantes.

In [9]:
df = df.sort_values("datetime").reset_index(drop=True)

duplicates_before = df.duplicated(subset="datetime").sum()

print(f"Date de début : {df['datetime'].min()}")
print(f"Date de fin   : {df['datetime'].max()}")
print(f"Doublons temporels détectés : {duplicates_before}")
print(f"Valeurs manquantes totales avant indexation : {df.isna().sum().sum()}")

Date de début : 2017-01-01 00:00:00
Date de fin   : 2017-12-30 23:50:00
Doublons temporels détectés : 0
Valeurs manquantes totales avant indexation : 0


## 10. Suppression des doublons temporels éventuels et indexation

Même si l’audit a montré qu’il n’existe pas de doublons temporels, cette étape sécurise le pipeline.

La colonne `datetime` devient l’index temporel du DataFrame.

In [10]:
df = df.drop_duplicates(subset="datetime", keep="first")
df = df.set_index("datetime").sort_index()

print(f"Dimensions après indexation : {df.shape}")
print(f"Fréquence brute inférée : {pd.infer_freq(df.index)}")

display(df.head())

Dimensions après indexation : (52416, 8)
Fréquence brute inférée : 10min


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power
datetime,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373
2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711
2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964


## 11. Création de la variable cible et de la charge totale

La cible principale du projet est la consommation de la Zone 1 :

`target = zone1_power`

La variable `total_load` représente la somme des consommations des trois zones :

`total_load = zone1_power + zone2_power + zone3_power`

La cible principale reste `target`, tandis que `total_load` sert d’indicateur complémentaire.

In [11]:
df["target"] = df["zone1_power"]

df["total_load"] = (
    df["zone1_power"] +
    df["zone2_power"] +
    df["zone3_power"]
)

print("Variables créées : target et total_load")
display(df[["zone1_power", "zone2_power", "zone3_power", "target", "total_load"]].head())

Variables créées : target et total_load


,zone1_power,zone2_power,zone3_power,target,total_load
datetime,,,,,
2017-01-01 00:00:00,34055.69620,16128.87538,20240.96386,34055.69620,70425.53544
2017-01-01 00:10:00,29814.68354,19375.07599,20131.08434,29814.68354,69320.84387
2017-01-01 00:20:00,29128.10127,19006.68693,19668.43373,29128.10127,67803.22193
2017-01-01 00:30:00,28228.86076,18361.09422,18899.27711,28228.86076,65489.23209
2017-01-01 00:40:00,27335.69620,17872.34043,18442.40964,27335.69620,63650.44627


## 12. Contrôle de cohérence de `target` et `total_load`

Cette cellule vérifie que :

- `target` est exactement égale à `zone1_power` ;
- `total_load` est bien la somme des trois zones.

In [12]:
target_check = np.allclose(df["target"], df["zone1_power"])

total_load_check = np.allclose(
    df["total_load"],
    df["zone1_power"] + df["zone2_power"] + df["zone3_power"]
)

print(f"Vérification target = zone1_power : {target_check}")
print(f"Vérification total_load = somme des trois zones : {total_load_check}")

if not target_check:
    raise ValueError("La variable target n'est pas égale à zone1_power.")

if not total_load_check:
    raise ValueError("La variable total_load est incorrecte.")

Vérification target = zone1_power : True
Vérification total_load = somme des trois zones : True


## 13. Rééchantillonnage horaire

Le dataset original est à fréquence de 10 minutes.

Pour la modélisation principale, les données sont rééchantillonnées à fréquence horaire.

La moyenne est utilisée car les colonnes de consommation sont traitées comme des niveaux moyens de charge sur les intervalles de mesure.

Ce choix permet :

- de réduire le bruit ;
- de simplifier la modélisation ;
- d’accélérer les modèles ;
- de conserver les cycles journaliers et hebdomadaires.

In [13]:
df_hourly = df.resample("1h").mean(numeric_only=True)

print(f"Dimensions après rééchantillonnage horaire : {df_hourly.shape}")
print(f"Fréquence horaire inférée : {pd.infer_freq(df_hourly.index)}")
print(f"Valeurs manquantes après resampling : {df_hourly.isna().sum().sum()}")

display(df_hourly.head())

Dimensions après rééchantillonnage horaire : (8736, 10)
Fréquence horaire inférée : h
Valeurs manquantes après resampling : 0


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load
datetime,,,,,,,,,,
2017-01-01 00:00:00,6.196833,75.066667,0.081833,0.063500,0.098833,29197.974683,18026.747720,19252.048193,29197.974683,66476.770597
2017-01-01 01:00:00,5.548833,77.583333,0.082000,0.056833,0.112500,24657.215190,16078.419453,17042.891567,24657.215190,57778.526210
2017-01-01 02:00:00,5.054333,78.933333,0.082333,0.063000,0.129167,22083.037973,14330.699088,15676.144578,22083.037973,52089.881640
2017-01-01 03:00:00,5.004333,77.083333,0.082833,0.059833,0.141000,20811.139240,13219.452887,14883.855422,20811.139240,48914.447548
2017-01-01 04:00:00,5.097667,74.050000,0.082333,0.058000,0.122833,20475.949367,12921.580547,14317.108433,20475.949367,47714.638347


## 14. Vérification de la grille horaire complète

Cette cellule vérifie si toutes les heures attendues sont bien présentes entre la première et la dernière date.

Cela permet de détecter d’éventuels trous temporels après rééchantillonnage.

In [14]:
expected_hourly_index = pd.date_range(
    start=df_hourly.index.min(),
    end=df_hourly.index.max(),
    freq="1h"
)

missing_hours = expected_hourly_index.difference(df_hourly.index)

print(f"Nombre d'heures attendues : {len(expected_hourly_index)}")
print(f"Nombre d'heures obtenues  : {len(df_hourly)}")
print(f"Heures manquantes         : {len(missing_hours)}")

if len(missing_hours) > 0:
    print("Exemples d'heures manquantes :")
    print(missing_hours[:10])
else:
    print("Aucune heure manquante détectée.")

Nombre d'heures attendues : 8736
Nombre d'heures obtenues  : 8736
Heures manquantes         : 0
Aucune heure manquante détectée.


## 14 bis. Contrôle du nombre de mesures 10 minutes par heure

La grille horaire peut être complète même si certaines heures sont calculées avec moins de 6 mesures de 10 minutes.

Ce contrôle vérifie donc que chaque heure du dataset brut contient exactement 6 observations avant le rééchantillonnage horaire.

Ce point renforce la fiabilité du resampling : une moyenne horaire calculée sur 6 mesures est plus fiable qu’une moyenne calculée sur 4 ou 5 mesures.

In [15]:
hourly_counts = df.resample("1h").size()
hourly_counts_distribution = hourly_counts.value_counts().sort_index()

print("Distribution du nombre de mesures 10 minutes par heure :")
display(hourly_counts_distribution.to_frame(name="nombre_d_heures"))

expected_measurements_per_hour = 6
incomplete_hours = hourly_counts[hourly_counts != expected_measurements_per_hour]

print(f"Nombre d'heures contrôlées                  : {len(hourly_counts)}")
print(f"Nombre attendu de mesures par heure         : {expected_measurements_per_hour}")
print(f"Nombre d'heures avec un nombre incorrect    : {len(incomplete_hours)}")

if len(incomplete_hours) > 0:
    print("Exemples d'heures incomplètes ou anormales :")
    display(incomplete_hours.head(10).to_frame(name="nb_mesures"))
    raise ValueError(
        "Certaines heures ne contiennent pas exactement 6 observations de 10 minutes. "
        "Le resampling horaire doit être inspecté avant de continuer."
    )

print("Contrôle validé : chaque heure contient exactement 6 mesures de 10 minutes.")

Distribution du nombre de mesures 10 minutes par heure :


,nombre_d_heures
6,8736


Nombre d'heures contrôlées                  : 8736
Nombre attendu de mesures par heure         : 6
Nombre d'heures avec un nombre incorrect    : 0
Contrôle validé : chaque heure contient exactement 6 mesures de 10 minutes.


## 15. Gestion sécurisée des valeurs manquantes

Le Data Audit a confirmé que le dataset brut ne contient aucune valeur manquante et aucune rupture temporelle. Dans ce notebook, aucune imputation réelle n’est donc nécessaire.

Une fonction d’imputation limitée est tout de même conservée pour rendre le pipeline robuste si une version future du fichier contient de petits trous.

La stratégie utilisée ici est volontairement prudente :

1. interpolation temporelle uniquement pour les petits trous ;
2. forward fill limité pour les valeurs restantes proches ;
3. suppression contrôlée des lignes seulement si des valeurs restent manquantes.

On évite un `backfill` global, car il peut utiliser une information future.

Pour un dataset réellement abîmé, avec des gaps moyens ou longs, il faudrait aller plus loin :

- gaps courts : interpolation temporelle ;
- gaps moyens : imputation saisonnière, par exemple même heure et même jour des semaines précédentes ;
- gaps longs : masquage ou exclusion des périodes concernées pour éviter une reconstruction artificielle de la série.

In [16]:
def handle_missing_values(data: pd.DataFrame, limit: int = 3) -> pd.DataFrame:
    """
    Gère les valeurs manquantes dans une série temporelle horaire.

    Paramètres
    ----------
    data : pd.DataFrame
        Dataset indexé par datetime.
    limit : int
        Nombre maximal de périodes consécutives à interpoler ou propager.

    Retour
    ------
    pd.DataFrame
        Dataset nettoyé.
    """
    data_clean = data.copy()

    missing_before = int(data_clean.isna().sum().sum())

    data_clean = data_clean.interpolate(
        method="time",
        limit=limit,
        limit_direction="forward"
    )

    data_clean = data_clean.ffill(limit=limit)

    missing_after = int(data_clean.isna().sum().sum())

    print(f"Valeurs manquantes avant traitement : {missing_before}")
    print(f"Valeurs manquantes après traitement : {missing_after}")

    if missing_after > 0:
        print("Certaines valeurs restent manquantes. Les lignes concernées seront supprimées.")
        data_clean = data_clean.dropna()
        print(f"Valeurs manquantes après suppression contrôlée : {data_clean.isna().sum().sum()}")

    return data_clean

In [17]:
df_clean = handle_missing_values(df_hourly, limit=3)

print(f"Dimensions après gestion des valeurs manquantes : {df_clean.shape}")

Valeurs manquantes avant traitement : 0
Valeurs manquantes après traitement : 0
Dimensions après gestion des valeurs manquantes : (8736, 10)


## 16. Contrôle des plages de valeurs

Cette cellule vérifie les valeurs physiquement incohérentes :

- consommation négative ;
- humidité hors intervalle `[0, 100]` ;
- température très éloignée d’une plage réaliste ;
- vitesse du vent négative.

Ces contrôles permettent de distinguer les valeurs plausibles des erreurs techniques.

In [18]:
range_checks = {
    "negative_zone1_power": int((df_clean["zone1_power"] < 0).sum()),
    "negative_zone2_power": int((df_clean["zone2_power"] < 0).sum()),
    "negative_zone3_power": int((df_clean["zone3_power"] < 0).sum()),
    "negative_target": int((df_clean["target"] < 0).sum()),
    "negative_total_load": int((df_clean["total_load"] < 0).sum()),
    "humidity_out_of_range": int((~df_clean["humidity"].between(0, 100)).sum()),
    "temperature_out_of_range": int((~df_clean["temperature"].between(-10, 55)).sum()),
    "negative_wind_speed": int((df_clean["wind_speed"] < 0).sum())
}

range_checks_df = pd.Series(range_checks).to_frame(name="count")

display(range_checks_df)

if range_checks_df["count"].sum() == 0:
    print("Aucune valeur physiquement incohérente détectée.")
else:
    print("Attention : certaines valeurs incohérentes doivent être inspectées.")

,count
negative_zone1_power,0
negative_zone2_power,0
negative_zone3_power,0
negative_target,0
negative_total_load,0
humidity_out_of_range,0
temperature_out_of_range,0
negative_wind_speed,0


Aucune valeur physiquement incohérente détectée.


## 17. Détection d’anomalies par Rolling Z-Score

Les séries temporelles énergétiques présentent naturellement des pics.

Il ne faut donc pas utiliser un Z-score global, car la consommation dépend fortement de l’heure, du jour et de la saison.

La méthode utilisée ici est un Z-score glissant :

- calcul d’une moyenne locale ;
- calcul d’un écart-type local ;
- comparaison de chaque valeur à son contexte récent.

Les anomalies détectées sont documentées, mais les pics de charge ne sont pas supprimés automatiquement.

In [19]:
def detect_rolling_outliers(
    series: pd.Series,
    window: int = 24,
    threshold: float = 3.5
) -> tuple[pd.Series, pd.Series]:
    """
    Détecte les anomalies avec un Z-score glissant basé sur le passé.

    Cette version évite d'utiliser les valeurs futures.

    Paramètres
    ----------
    series : pd.Series
        Série temporelle.
    window : int
        Taille de la fenêtre glissante.
    threshold : float
        Seuil du Z-score absolu.

    Retour
    ------
    outlier_mask : pd.Series
        Masque booléen des anomalies.
    zscore : pd.Series
        Valeurs du Z-score glissant.
    """
    past_series = series.shift(1)

    rolling_mean = past_series.rolling(
        window=window,
        min_periods=max(3, window // 2)
    ).mean()

    rolling_std = past_series.rolling(
        window=window,
        min_periods=max(3, window // 2)
    ).std()

    zscore = ((series - rolling_mean) / rolling_std.replace(0, np.nan)).abs()

    outlier_mask = zscore > threshold
    outlier_mask = outlier_mask.fillna(False)

    return outlier_mask, zscore

## 18. Application de la détection d’anomalies

La détection est appliquée aux variables météo, aux consommations des trois zones, à la cible et à la charge totale.

Le résultat donne un résumé du nombre d’anomalies détectées par variable.

In [20]:
cols_to_check = [
    "temperature",
    "humidity",
    "wind_speed",
    "general_diffuse_flows",
    "diffuse_flows",
    "zone1_power",
    "zone2_power",
    "zone3_power",
    "target",
    "total_load"
]

outlier_summary = {}

for col in cols_to_check:
    mask, zscore = detect_rolling_outliers(
        df_clean[col],
        window=24,
        threshold=3.5
    )
    outlier_summary[col] = int(mask.sum())

outlier_summary_df = (
    pd.Series(outlier_summary)
    .sort_values(ascending=False)
    .to_frame(name="outlier_count")
)

display(outlier_summary_df)

,outlier_count
wind_speed,259
diffuse_flows,84
humidity,67
general_diffuse_flows,40
temperature,38
zone3_power,1
zone1_power,0
zone2_power,0
target,0
total_load,0


## 19. Annotation des pics de charge

Les pics de charge sont importants pour le projet.

Ils ne sont donc pas supprimés automatiquement.

À la place, on crée des colonnes d’annotation :

- `is_target_outlier` ;
- `is_total_load_outlier` ;
- `is_load_outlier`.

Ces colonnes servent à documenter les périodes inhabituelles.

Attention : ces colonnes ne doivent être utilisées comme variables explicatives que si leur construction est compatible avec le scénario de prédiction.

In [21]:
target_outliers, target_zscore = detect_rolling_outliers(
    df_clean["target"],
    window=24,
    threshold=3.5
)

total_load_outliers, total_load_zscore = detect_rolling_outliers(
    df_clean["total_load"],
    window=24,
    threshold=3.5
)

df_clean["target_rolling_zscore"] = target_zscore
df_clean["total_load_rolling_zscore"] = total_load_zscore

df_clean["is_target_outlier"] = target_outliers.astype(int)
df_clean["is_total_load_outlier"] = total_load_outliers.astype(int)

df_clean["is_load_outlier"] = (
    (df_clean["is_target_outlier"] == 1) |
    (df_clean["is_total_load_outlier"] == 1)
).astype(int)

print(f"Pics/anomalies sur target détectés : {df_clean['is_target_outlier'].sum()}")
print(f"Pics/anomalies sur total_load détectés : {df_clean['is_total_load_outlier'].sum()}")
print(f"Pics/anomalies de charge globalement annotés : {df_clean['is_load_outlier'].sum()}")

Pics/anomalies sur target détectés : 0
Pics/anomalies sur total_load détectés : 0
Pics/anomalies de charge globalement annotés : 0


## 19 bis. Correction des `NaN` créés par les Rolling Z-Scores

Les colonnes `target_rolling_zscore` et `total_load_rolling_zscore` utilisent une fenêtre glissante basée sur le passé.  
Au début de la série, il n'existe pas encore assez d'historique pour calculer la moyenne et l'écart-type locaux.

Ces `NaN` ne proviennent donc pas du fichier brut : ils sont créés par le prétraitement.  
Pour conserver un dataset propre avant sauvegarde, ils sont remplacés par `0`, ce qui signifie : **aucun écart local détecté au début de la série**.

In [22]:
# Correction des NaN créés par les rolling z-scores au début de la série

zscore_cols = [
    "target_rolling_zscore",
    "total_load_rolling_zscore"
]

print("Valeurs manquantes dans les colonnes z-score avant correction :")
display(df_clean[zscore_cols].isna().sum().to_frame(name="missing_count"))

df_clean[zscore_cols] = df_clean[zscore_cols].fillna(0)

print("Valeurs manquantes dans les colonnes z-score après correction :")
display(df_clean[zscore_cols].isna().sum().to_frame(name="missing_count"))

print(f"Valeurs manquantes totales après correction : {df_clean.isna().sum().sum()}")

Valeurs manquantes dans les colonnes z-score avant correction :


,missing_count
target_rolling_zscore,12
total_load_rolling_zscore,12


Valeurs manquantes dans les colonnes z-score après correction :


,missing_count
target_rolling_zscore,0
total_load_rolling_zscore,0


Valeurs manquantes totales après correction : 0


## 20. Vérification finale du dataset propre

Avant la sauvegarde, on vérifie :

- les dimensions finales ;
- les types de variables ;
- les valeurs manquantes restantes ;
- la période couverte ;
- la fréquence inférée.

In [23]:
print("Résumé du dataset propre :")
print(f"Dimensions : {df_clean.shape}")
print(f"Date début : {df_clean.index.min()}")
print(f"Date fin   : {df_clean.index.max()}")
print(f"Fréquence inférée : {pd.infer_freq(df_clean.index)}")
print(f"Valeurs manquantes restantes : {df_clean.isna().sum().sum()}")

display(df_clean.head())
display(df_clean.tail())

Résumé du dataset propre :
Dimensions : (8736, 15)
Date début : 2017-01-01 00:00:00
Date fin   : 2017-12-30 23:00:00
Fréquence inférée : h
Valeurs manquantes restantes : 0


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load,target_rolling_zscore,total_load_rolling_zscore,is_target_outlier,is_total_load_outlier,is_load_outlier
datetime,,,,,,,,,,,,,,,
2017-01-01 00:00:00,6.196833,75.066667,0.081833,0.063500,0.098833,29197.974683,18026.747720,19252.048193,29197.974683,66476.770597,0.0,0.0,0,0,0
2017-01-01 01:00:00,5.548833,77.583333,0.082000,0.056833,0.112500,24657.215190,16078.419453,17042.891567,24657.215190,57778.526210,0.0,0.0,0,0,0
2017-01-01 02:00:00,5.054333,78.933333,0.082333,0.063000,0.129167,22083.037973,14330.699088,15676.144578,22083.037973,52089.881640,0.0,0.0,0,0,0
2017-01-01 03:00:00,5.004333,77.083333,0.082833,0.059833,0.141000,20811.139240,13219.452887,14883.855422,20811.139240,48914.447548,0.0,0.0,0,0,0
2017-01-01 04:00:00,5.097667,74.050000,0.082333,0.058000,0.122833,20475.949367,12921.580547,14317.108433,20475.949367,47714.638347,0.0,0.0,0,0,0


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load,target_rolling_zscore,total_load_rolling_zscore,is_target_outlier,is_total_load_outlier,is_load_outlier
datetime,,,,,,,,,,,,,,,
2017-12-30 19:00:00,9.453333,62.406667,0.074667,0.052000,0.102000,37513.814957,32497.698680,16926.770708,37513.814957,86938.284345,1.517436,1.611855,0,0,0
2017-12-30 20:00:00,9.041667,63.990000,0.080333,0.052667,0.105000,37008.871988,32020.251610,16998.799520,37008.871988,86027.923118,1.454852,1.558531,0,0,0
2017-12-30 21:00:00,8.011667,69.675000,0.081500,0.073167,0.098333,35358.174905,30757.901197,16608.883553,35358.174905,82724.959655,1.204273,1.339369,0,0,0
2017-12-30 22:00:00,7.598333,70.315000,0.081833,0.058667,0.108167,33993.409380,28477.447070,15614.885955,33993.409380,78085.742405,0.995139,1.021828,0,0,0
2017-12-30 23:00:00,6.877500,72.900000,0.081500,0.060333,0.092667,30107.984788,25713.409022,14143.577428,30107.984788,69964.971238,0.331566,0.434969,0,0,0


## 20 bis. Vérification finale stricte avant sauvegarde

Cette cellule bloque l'exécution si le dataset n'est pas réellement propre.  
Elle garantit que le notebook ne peut pas sauvegarder un fichier `clean` contenant encore des valeurs manquantes, des doublons temporels ou une fréquence non horaire.

In [24]:
# Vérification finale stricte avant split et sauvegarde

def is_hourly_frequency(freq: str | None) -> bool:
    """
    Vérifie que la fréquence inférée correspond à une fréquence horaire.
    Compatible avec les notations Pandas récentes et anciennes : 'h', 'H', '1h', '1H'.
    """
    return str(freq) in {"h", "H", "1h", "1H"}

missing_final = int(df_clean.isna().sum().sum())
final_freq = pd.infer_freq(df_clean.index)
duplicated_final = int(df_clean.index.duplicated().sum())

print(f"Valeurs manquantes finales avant sauvegarde : {missing_final}")
print(f"Doublons temporels finaux                 : {duplicated_final}")
print(f"Fréquence finale inférée                  : {final_freq}")

if missing_final > 0:
    missing_by_col = df_clean.isna().sum()
    missing_by_col = missing_by_col[missing_by_col > 0]
    display(missing_by_col.to_frame(name="missing_count"))
    raise ValueError("Le dataset contient encore des valeurs manquantes. Correction obligatoire avant sauvegarde.")

if duplicated_final > 0:
    raise ValueError("Le dataset contient des doublons temporels.")

if not is_hourly_frequency(final_freq):
    raise ValueError(f"Fréquence horaire invalide : {final_freq}")

print("Dataset propre validé : aucune valeur manquante, aucun doublon, fréquence horaire correcte.")

Valeurs manquantes finales avant sauvegarde : 0
Doublons temporels finaux                 : 0
Fréquence finale inférée                  : h
Dataset propre validé : aucune valeur manquante, aucun doublon, fréquence horaire correcte.


## 21. Split chronologique train / validation / test

Pour les séries temporelles, il est interdit de mélanger aléatoirement les observations.

Le split doit respecter l’ordre temporel :

- train : passé ;
- validation : période intermédiaire ;
- test : période future.

Ce découpage évite la fuite d’information du futur vers le passé.

In [25]:
def temporal_train_val_test_split(
    data: pd.DataFrame,
    train_size: float = 0.70,
    val_size: float = 0.15
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Sépare un dataset temporel en train, validation et test.

    Le découpage respecte strictement l'ordre chronologique.

    Paramètres
    ----------
    data : pd.DataFrame
        Dataset temporel.
    train_size : float
        Proportion du train.
    val_size : float
        Proportion de validation.

    Retour
    ------
    train_df, val_df, test_df
    """
    if train_size <= 0 or val_size <= 0 or train_size + val_size >= 1:
        raise ValueError("Les proportions de split sont invalides.")

    n = len(data)

    train_end = int(n * train_size)
    val_end = int(n * (train_size + val_size))

    train_df = data.iloc[:train_end].copy()
    val_df = data.iloc[train_end:val_end].copy()
    test_df = data.iloc[val_end:].copy()

    print("Split chronologique créé :")
    print(f"Train : {train_df.index.min()} → {train_df.index.max()} | {train_df.shape}")
    print(f"Val   : {val_df.index.min()} → {val_df.index.max()} | {val_df.shape}")
    print(f"Test  : {test_df.index.min()} → {test_df.index.max()} | {test_df.shape}")

    return train_df, val_df, test_df

In [26]:
train_df, val_df, test_df = temporal_train_val_test_split(
    df_clean,
    train_size=0.70,
    val_size=0.15
)

Split chronologique créé :
Train : 2017-01-01 00:00:00 → 2017-09-12 18:00:00 | (6115, 15)
Val   : 2017-09-12 19:00:00 → 2017-11-06 08:00:00 | (1310, 15)
Test  : 2017-11-06 09:00:00 → 2017-12-30 23:00:00 | (1311, 15)


## 22. Normalisation sans data leakage

La normalisation est importante pour certains modèles, notamment le GRU.

Règle essentielle :

- le scaler est ajusté uniquement sur le train ;
- le scaler est ensuite appliqué à validation et test.

Cela évite la fuite d’information du futur vers le passé.

Remarque : après le feature engineering, les scalers devront être réajustés sur le dataset enrichi.

In [27]:
TARGET_COL = "target"

# Colonnes interdites comme variables explicatives directes pour une prédiction future.
# Elles sont conservées dans les fichiers propres pour audit, comparaison et feature engineering,
# mais elles ne doivent pas être utilisées au même timestamp pour prédire target.
FORBIDDEN_DIRECT_FEATURES = [
    # Cible directe ou consommations instantanées
    "target",
    "zone1_power",
    "zone2_power",
    "zone3_power",
    "total_load",

    # Colonnes de diagnostic calculées avec la valeur actuelle de charge
    "target_rolling_zscore",
    "total_load_rolling_zscore",
    "is_target_outlier",
    "is_total_load_outlier",
    "is_load_outlier"
]

# Features autorisées dans ce preprocessing de base.
# Les lags, rolling features basées sur le passé et variables calendaires seront créés dans le notebook suivant.
BASE_FEATURES_ALLOWED = [
    "temperature",
    "humidity",
    "wind_speed",
    "general_diffuse_flows",
    "diffuse_flows"
]

missing_allowed_features = sorted(set(BASE_FEATURES_ALLOWED) - set(df_clean.columns))
if missing_allowed_features:
    raise ValueError(f"Features météo attendues absentes du dataset : {missing_allowed_features}")

feature_cols = BASE_FEATURES_ALLOWED.copy()

print(f"Nombre de variables explicatives utilisées pour le scaling de base : {len(feature_cols)}")
print("Variables explicatives utilisées :")
print(feature_cols)

# Contrôle anti-leakage explicite
forbidden_in_features = sorted(set(FORBIDDEN_DIRECT_FEATURES).intersection(feature_cols))

if forbidden_in_features:
    raise ValueError(f"Data leakage détecté dans feature_cols : {forbidden_in_features}")

print("Contrôle anti-leakage validé :")
print("- aucune consommation instantanée n'est utilisée comme feature directe ;")
print("- aucun z-score de charge ni indicateur d'outlier de charge n'est utilisé comme feature directe ;")
print("- seules les variables météo de base sont normalisées dans ce notebook.")

print("\nRègle pour le notebook de feature engineering :")
print("Utiliser les consommations uniquement sous forme de lags ou rolling features basées sur le passé.")

Nombre de variables explicatives utilisées pour le scaling de base : 5
Variables explicatives utilisées :
['temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows']
Contrôle anti-leakage validé :
- aucune consommation instantanée n'est utilisée comme feature directe ;
- aucun z-score de charge ni indicateur d'outlier de charge n'est utilisé comme feature directe ;
- seules les variables météo de base sont normalisées dans ce notebook.

Règle pour le notebook de feature engineering :
Utiliser les consommations uniquement sous forme de lags ou rolling features basées sur le passé.


## 22 bis. Contrat de sécurité pour le notebook de Feature Engineering

Les fichiers `train_clean.csv`, `val_clean.csv` et `test_clean.csv` conservent volontairement les colonnes de consommation instantanée.

Ce choix est utile pour créer les lags et les statistiques glissantes dans le notebook suivant. Cependant, ces colonnes ne doivent jamais être utilisées directement au même timestamp comme variables explicatives.

Règle à respecter dans `04_feature_engineering.ipynb` :

- autorisé : `target_lag_1h`, `target_lag_24h`, `target_lag_168h`, rolling mean/std calculés avec `shift(1)` ;
- autorisé : variables météo et variables calendaires ;
- interdit : `zone1_power`, `target`, `total_load` au même timestamp ;
- interdit : z-scores de charge et indicateurs d’outliers de charge comme features prédictives directes.

In [28]:
feature_engineering_rules = {
    "allowed_consumption_usage": [
        "target_lag_1h",
        "target_lag_24h",
        "target_lag_168h",
        "rolling_features_with_shift_1"
    ],
    "allowed_other_features": [
        "weather_features",
        "calendar_features",
        "cyclical_time_encoding",
        "holiday_features_if_available"
    ],
    "forbidden_direct_features": FORBIDDEN_DIRECT_FEATURES,
    "main_rule": (
        "Les colonnes de consommation instantanée sont conservées pour construire "
        "des lags et rolling features, mais ne doivent pas être utilisées directement "
        "comme variables explicatives au même timestamp."
    )
}

display(pd.Series(feature_engineering_rules, name="rule"))

allowed_consumption_usage    [target_lag_1h, target_lag_24h, target_lag_168...
allowed_other_features       [weather_features, calendar_features, cyclical...
forbidden_direct_features    [target, zone1_power, zone2_power, zone3_power...
main_rule                    Les colonnes de consommation instantanée sont ...
Name: rule, dtype: object

In [29]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(train_df[feature_cols])
X_val_scaled = scaler_X.transform(val_df[feature_cols])
X_test_scaled = scaler_X.transform(test_df[feature_cols])

y_train_scaled = scaler_y.fit_transform(train_df[[TARGET_COL]])
y_val_scaled = scaler_y.transform(val_df[[TARGET_COL]])
y_test_scaled = scaler_y.transform(test_df[[TARGET_COL]])

print("Normalisation terminée sans data leakage.")
print(f"X_train_scaled shape : {X_train_scaled.shape}")
print(f"X_val_scaled shape   : {X_val_scaled.shape}")
print(f"X_test_scaled shape  : {X_test_scaled.shape}")

Normalisation terminée sans data leakage.
X_train_scaled shape : (6115, 5)
X_val_scaled shape   : (1310, 5)
X_test_scaled shape  : (1311, 5)


## 23. Conversion des données normalisées en DataFrame

Les tableaux normalisés sont convertis en DataFrame afin de conserver :

- les noms des colonnes ;
- les index temporels ;
- la lisibilité des fichiers sauvegardés.

In [30]:
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    index=train_df.index,
    columns=feature_cols
)

X_val_scaled_df = pd.DataFrame(
    X_val_scaled,
    index=val_df.index,
    columns=feature_cols
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    index=test_df.index,
    columns=feature_cols
)

y_train_scaled_df = pd.DataFrame(
    y_train_scaled,
    index=train_df.index,
    columns=[TARGET_COL]
)

y_val_scaled_df = pd.DataFrame(
    y_val_scaled,
    index=val_df.index,
    columns=[TARGET_COL]
)

y_test_scaled_df = pd.DataFrame(
    y_test_scaled,
    index=test_df.index,
    columns=[TARGET_COL]
)

display(X_train_scaled_df.head())
display(y_train_scaled_df.head())

,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows
datetime,,,,,
2017-01-01 00:00:00,-2.112482,0.462947,-0.815577,-0.735586,-0.661449
2017-01-01 01:00:00,-2.217439,0.620177,-0.815506,-0.735609,-0.661344
2017-01-01 02:00:00,-2.297534,0.704519,-0.815364,-0.735588,-0.661217
2017-01-01 03:00:00,-2.305632,0.588939,-0.815151,-0.735599,-0.661126
2017-01-01 04:00:00,-2.290515,0.399430,-0.815364,-0.735605,-0.661265


,target
datetime,
2017-01-01 00:00:00,-0.522653
2017-01-01 01:00:00,-1.154081
2017-01-01 02:00:00,-1.512040
2017-01-01 03:00:00,-1.688908
2017-01-01 04:00:00,-1.735518


## 24. Sauvegarde du dataset propre

Cette cellule sauvegarde :

- le dataset horaire complet nettoyé ;
- le train propre ;
- la validation propre ;
- le test propre.

Ces fichiers seront utilisés dans le notebook suivant de feature engineering.

In [31]:
clean_full_path = PROCESSED_DIR / "tetouan_hourly_clean.csv"
train_path = PROCESSED_DIR / "train_clean.csv"
val_path = PROCESSED_DIR / "val_clean.csv"
test_path = PROCESSED_DIR / "test_clean.csv"

df_clean.to_csv(clean_full_path, index=True)
train_df.to_csv(train_path, index=True)
val_df.to_csv(val_path, index=True)
test_df.to_csv(test_path, index=True)

print("Fichiers propres sauvegardés :")
print(f"- {clean_full_path}")
print(f"- {train_path}")
print(f"- {val_path}")
print(f"- {test_path}")

Fichiers propres sauvegardés :
- D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\tetouan_hourly_clean.csv
- D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\train_clean.csv
- D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\val_clean.csv
- D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\data\processed\test_clean.csv


## 25. Sauvegarde des fichiers normalisés de base

Ces fichiers normalisés sont utiles pour vérifier le pipeline.

Cependant, pour la modélisation finale, il faudra refaire la normalisation après le feature engineering, car de nouvelles variables seront ajoutées.

In [32]:
X_train_scaled_path = PROCESSED_DIR / "X_train_scaled_base.csv"
X_val_scaled_path = PROCESSED_DIR / "X_val_scaled_base.csv"
X_test_scaled_path = PROCESSED_DIR / "X_test_scaled_base.csv"

y_train_scaled_path = PROCESSED_DIR / "y_train_scaled_base.csv"
y_val_scaled_path = PROCESSED_DIR / "y_val_scaled_base.csv"
y_test_scaled_path = PROCESSED_DIR / "y_test_scaled_base.csv"

X_train_scaled_df.to_csv(X_train_scaled_path, index=True)
X_val_scaled_df.to_csv(X_val_scaled_path, index=True)
X_test_scaled_df.to_csv(X_test_scaled_path, index=True)

y_train_scaled_df.to_csv(y_train_scaled_path, index=True)
y_val_scaled_df.to_csv(y_val_scaled_path, index=True)
y_test_scaled_df.to_csv(y_test_scaled_path, index=True)

print("Fichiers normalisés de base sauvegardés.")

Fichiers normalisés de base sauvegardés.


## 26. Sauvegarde des scalers

Les scalers sont sauvegardés pour pouvoir appliquer exactement la même transformation plus tard.

Important : ces scalers sont ajustés uniquement sur le train.

In [33]:
scaler_X_path = MODELS_DIR / "scaler_X_base.pkl"
scaler_y_path = MODELS_DIR / "scaler_y_base.pkl"

joblib.dump(scaler_X, scaler_X_path)
joblib.dump(scaler_y, scaler_y_path)

print("Scalers sauvegardés :")
print(f"- {scaler_X_path}")
print(f"- {scaler_y_path}")

Scalers sauvegardés :
- D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\models\scaler_X_base.pkl
- D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\models\scaler_y_base.pkl


## 27. Rapport de preprocessing

Cette cellule crée un résumé automatique du preprocessing sous forme de dictionnaire.

Le rapport est sauvegardé en JSON dans `results/preprocessing`.

In [34]:
preprocessing_report = {
    "raw_shape": list(df_raw.shape),
    "clean_hourly_shape": list(df_clean.shape),
    "train_shape": list(train_df.shape),
    "val_shape": list(val_df.shape),
    "test_shape": list(test_df.shape),

    "raw_start_date": str(df.index.min()),
    "raw_end_date": str(df.index.max()),
    "hourly_start_date": str(df_clean.index.min()),
    "hourly_end_date": str(df_clean.index.max()),

    "raw_inferred_frequency": str(pd.infer_freq(df.index)),
    "hourly_inferred_frequency": str(pd.infer_freq(df_clean.index)),

    "invalid_dates": int(invalid_dates),
    "duplicates_before": int(duplicates_before),
    "duplicates_final": int(df_clean.index.duplicated().sum()),
    "missing_after_cleaning": int(df_clean.isna().sum().sum()),
    "project_root": str(PROJECT_ROOT),
    "raw_path": str(RAW_PATH),

    "missing_hours": int(len(missing_hours)),
    "hourly_counts_distribution": {str(k): int(v) for k, v in hourly_counts_distribution.items()},
    "incomplete_hours_after_10min_check": int(len(incomplete_hours)),
    "expected_measurements_per_hour": int(expected_measurements_per_hour),

    "range_checks": {key: int(value) for key, value in range_checks.items()},
    "outlier_summary": {key: int(value) for key, value in outlier_summary.items()},

    "target_outliers": int(df_clean["is_target_outlier"].sum()),
    "total_load_outliers": int(df_clean["is_total_load_outlier"].sum()),
    "load_outliers": int(df_clean["is_load_outlier"].sum()),

    "target_column": TARGET_COL,
    "forbidden_direct_features": FORBIDDEN_DIRECT_FEATURES,
    "feature_columns_base": feature_cols,

    "strict_validation": {
        "missing_final": int(df_clean.isna().sum().sum()),
        "duplicated_final": int(df_clean.index.duplicated().sum()),
        "final_frequency": str(pd.infer_freq(df_clean.index)),
        "anti_leakage_checked": True
    },

    "saved_files": {
        "clean_full": str(clean_full_path),
        "train_clean": str(train_path),
        "val_clean": str(val_path),
        "test_clean": str(test_path),
        "scaler_X_base": str(scaler_X_path),
        "scaler_y_base": str(scaler_y_path)
    }
}

report_path = RESULTS_DIR / "preprocessing_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(preprocessing_report, f, indent=4, ensure_ascii=False)

print(f"Rapport de preprocessing sauvegardé : {report_path}")

Rapport de preprocessing sauvegardé : D:\UB\0 Inbox\00-projects-code\dm-pfm-project\smart-city-energy-forecasting-tetouan\results\preprocessing\preprocessing_report.json


## 28. Résumé final du preprocessing

Cette dernière cellule affiche un résumé lisible des résultats principaux obtenus.

In [35]:
print("=" * 80)
print("RÉSUMÉ FINAL DU PREPROCESSING")
print("=" * 80)

print(f"Dimensions brutes                      : {df_raw.shape}")
print(f"Dimensions après resampling horaire     : {df_hourly.shape}")
print(f"Dimensions finales propres              : {df_clean.shape}")

print("-" * 80)
print(f"Période horaire                         : {df_clean.index.min()} → {df_clean.index.max()}")
print(f"Fréquence finale                        : {pd.infer_freq(df_clean.index)}")

print("-" * 80)
print(f"Valeurs manquantes finales              : {df_clean.isna().sum().sum()}")
print(f"Heures manquantes                       : {len(missing_hours)}")
print(f"Heures avec nb mesures 10min incorrect : {len(incomplete_hours)}")
print(f"Doublons temporels initiaux             : {duplicates_before}")
print(f"Doublons temporels finaux               : {df_clean.index.duplicated().sum()}")

print("-" * 80)
print(f"Anomalies target conservées             : {df_clean['is_target_outlier'].sum()}")
print(f"Anomalies total_load conservées         : {df_clean['is_total_load_outlier'].sum()}")

print("-" * 80)
print(f"Variables explicatives scaling base     : {feature_cols}")

print("-" * 80)
print(f"Train                                   : {train_df.shape} | {train_df.index.min()} → {train_df.index.max()}")
print(f"Validation                              : {val_df.shape} | {val_df.index.min()} → {val_df.index.max()}")
print(f"Test                                    : {test_df.shape} | {test_df.index.min()} → {test_df.index.max()}")

print("-" * 80)
print("Fichiers principaux sauvegardés :")
print(f"- {clean_full_path}")
print(f"- {train_path}")
print(f"- {val_path}")
print(f"- {test_path}")
print(f"- {report_path}")

print("=" * 80)
print("Prétraitement terminé avec succès.")

RÉSUMÉ FINAL DU PREPROCESSING
Dimensions brutes                      : (52416, 9)
Dimensions après resampling horaire     : (8736, 10)
Dimensions finales propres              : (8736, 15)
--------------------------------------------------------------------------------
Période horaire                         : 2017-01-01 00:00:00 → 2017-12-30 23:00:00
Fréquence finale                        : h
--------------------------------------------------------------------------------
Valeurs manquantes finales              : 0
Heures manquantes                       : 0
Heures avec nb mesures 10min incorrect : 0
Doublons temporels initiaux             : 0
Doublons temporels finaux               : 0
--------------------------------------------------------------------------------
Anomalies target conservées             : 0
Anomalies total_load conservées         : 0
--------------------------------------------------------------------------------
Variables explicatives scaling base     : ['temperatu

# Conclusion du notebook

Le notebook de preprocessing final a permis de transformer le dataset brut en un dataset horaire propre et exploitable.

Les garanties principales sont :

- chargement robuste du fichier depuis la racine du projet ou le dossier `notebooks/` ;
- colonnes renommées et standardisées ;
- dates et variables numériques correctement converties ;
- index temporel trié et régulier ;
- fréquence brute de 10 minutes contrôlée ;
- rééchantillonnage horaire validé ;
- vérification que chaque heure contient exactement 6 mesures de 10 minutes ;
- absence de valeurs manquantes finales ;
- absence de doublons temporels finaux ;
- anomalies conservées comme information métier ;
- split chronologique train / validation / test ;
- normalisation effectuée uniquement après split ;
- scalers ajustés uniquement sur le train ;
- exclusion stricte des colonnes de consommation instantanée et des z-scores de charge des features de base.

Ce notebook est donc validé comme version finale de preprocessing et peut être utilisé comme entrée du notebook `04_feature_engineering.ipynb`.